# 두 프롬프트로 KV 캐시의 메모리 사용량 재기

서빙 프로세스에서 키와 값에 쓸 수 있는 메모리가 120 MiB 남았습니다. 요청 네 개가 각각 512토큰을 입력하고 이후 256토큰을 더 저장할 수 있다고 할 때, 네 요청을 모두 받아도 될까요? 먼저 답을 예상해 보세요.

이 실습에서는 사전학습 모델을 CPU에서 실행하고, 캐시 텐서의 원소 수와 원소당 바이트 수를 직접 읽습니다. 패딩이 차지한 공간을 측정하고, 토큰을 하나씩 추가하며 캐시를 재사용한 결과를 전체 재계산과 비교합니다.

PyTorch가 설치된 Python 노트북이 필요합니다. 첫 셀은 Transformers를 설치하며, 체크포인트를 처음 내려받을 때 약 270 MB를 전송합니다. 이 실습의 FP32 모델 가중치만 약 514 MiB를 차지합니다. 검증 환경은 Python 3.12, CPU PyTorch 2.15.0, Transformers 5.24.1입니다.

Seongeun So의 《파운데이션 모델 엔지니어링》을 위해 Codex의 도움을 받아 작성했습니다. [한국어 원문과 계산기](https://sungeuns.github.io/foundation-model-engineering/ko/chapter-12/kv-cache-management/#kv-payload-notebook) · [소스 저장소](https://github.com/sungeuns/foundation-model-engineering)

In [ ]:
%pip install -q "transformers==5.24.1"

## 체크포인트에서 헤드 수 읽기

[SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)은 여러 쿼리 헤드가 같은 키와 값을 읽는 그룹 쿼리 어텐션(GQA)을 사용합니다. 모델과 토크나이저를 같은 리비전에 고정하고 CPU에서 FP32 평가 모드로 실행합니다. 다음 셀에서 쿼리 헤드 수와 KV 헤드 수 중 어느 수가 바이트 계산에 들어가는지 확인해 보세요.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

model_id = "HuggingFaceTB/SmolLM2-135M"
revision = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, dtype=torch.float32, attn_implementation="eager"
).eval().cpu()
cfg = model.config
layers = cfg.num_hidden_layers
kv_heads = cfg.num_key_value_heads
head_dim = cfg.head_dim
value_bytes = next(model.parameters()).element_size()
bytes_per_position = 2 * layers * kv_heads * head_dim * value_bytes
print(f"{layers} layers; {cfg.num_attention_heads} query heads; {kv_heads} KV heads")
print(f"{head_dim} values per head; {value_bytes} bytes per value")
print(f"{bytes_per_position:,} bytes per stored position per request")

식 앞의 2는 키와 값 두 텐서를 뜻합니다. 같은 길이의 슬롯을 쓰는 요청이 `B`개이고 요청마다 `T`자리씩 저장한다면 이 모델의 캐시 텐서는 총 `B × T × bytes_per_position`바이트를 차지합니다. KV 헤드 3개 대신 쿼리 헤드 9개를 넣으면 세 배로 과대 추정합니다.

## 길이가 다른 요청을 처음 처리할 때 재기

두 프롬프트의 토큰 수가 다릅니다. 짧은 쪽의 왼쪽을 패딩해 직사각형 배치를 만들고, 마스크로 실제 텍스트의 위치를 표시합니다. 각 요청의 첫 실제 토큰이 위치 0을 갖도록 위치 ID도 마스크에서 구합니다. 두 프롬프트를 영어로 유지해야 양쪽 노트북에서 같은 토큰열을 비교할 수 있습니다.

In [ ]:
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
prompts = ["Water boils at", "When a kettle of water starts to boil, the steam"]
encoded = tokenizer(prompts, return_tensors="pt", padding=True)
ids, mask = encoded.input_ids, encoded.attention_mask
batch, width = ids.shape

def position_ids(mask):
    return (mask.cumsum(dim=-1) - 1).clamp_min(0)

with torch.inference_mode():
    output = model(
        input_ids=ids, attention_mask=mask, position_ids=position_ids(mask),
        use_cache=True, logits_to_keep=1,
    )

cache = output.past_key_values
assert isinstance(cache, DynamicCache)
print("Unpadded prompt lengths:", mask.sum(dim=1).tolist())
print("First layer's key shape:", tuple(cache.layers[0].keys.shape))

각 어텐션 층에는 `[배치, KV 헤드, 저장 위치, 헤드 차원]` 모양의 텐서 두 개가 있습니다. 모든 층에서 원소 수와 실제 원소 크기를 읽어 합계를 냅니다. 앞에서 예측한 값과 비교하면 쿼리 헤드 수를 잘못 넣었거나 디스크에 저장된 가중치의 자료형을 실행 중인 캐시의 자료형으로 착각한 경우를 발견할 수 있습니다.

In [ ]:
def measure_cache(cache, positions):
    assert len(cache.layers) == layers
    measured = 0
    for layer in cache.layers:
        for tensor in (layer.keys, layer.values):
            assert tensor.shape == (batch, kv_heads, positions, head_dim)
            measured += tensor.numel() * tensor.element_size()
    expected = batch * positions * bytes_per_position
    assert measured == expected, (measured, expected)
    return measured

prefill_bytes = measure_cache(cache, width)
padding_positions = batch * width - mask.sum().item()
print(f"Prompt cache: {prefill_bytes:,} bytes")
print(f"Of these, {padding_positions * bytes_per_position:,} bytes occupy padding positions")

마스크는 패딩이 어텐션 결과에 기여하는 것을 막지만 이 밀집 캐시에서는 패딩 위치에도 슬롯을 할당합니다. 프롬프트를 바꿔 보고 싶다면 두 문장 중 **긴 쪽의 토큰 수**로 먼저 용량을 예측한 후 다시 실행하세요.

## 네 토큰을 차례로 추가하며 로짓 비교하기

프롬프트를 처리하면 첫 생성 토큰을 고를 로짓이 나옵니다. 토큰을 고르는 순간에는 캐시가 늘지 않습니다. 고른 토큰을 모델에 다시 입력할 때 한 자리가 추가됩니다. 캐시를 쓰는 호출에는 새 토큰 하나와 지금까지의 전체 마스크를 넣습니다. 캐시를 끈 호출에는 전체 토큰열을 넣고 두 호출의 다음 토큰 로짓 벡터를 비교합니다. 이 과정을 네 번 반복합니다.

In [ ]:
next_ids = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
with torch.inference_mode():
    for step in range(1, 5):
        ids = torch.cat([ids, next_ids], dim=1)
        mask = torch.cat([mask, torch.ones_like(next_ids)], dim=1)
        positions = position_ids(mask)
        cached = model(
            input_ids=next_ids, attention_mask=mask, position_ids=positions[:, -1:],
            past_key_values=cache, use_cache=True, logits_to_keep=1,
        )
        cache = cached.past_key_values
        reference = model(
            input_ids=ids, attention_mask=mask, position_ids=positions,
            use_cache=False, logits_to_keep=1,
        )
        torch.testing.assert_close(cached.logits, reference.logits, rtol=1e-4, atol=1e-4)
        measured = measure_cache(cache, width + step)
        delta = (cached.logits - reference.logits).abs().max().item()
        print(f"+{step}: {measured:,} cache bytes; largest logit difference {delta:.2e}")
        next_ids = cached.logits[:, -1, :].argmax(dim=-1, keepdim=True)

두 방식은 행렬 모양이 달라서 부동소수점 결과가 조금 다를 수 있습니다. 반면 캐시 크기의 증가량은 정확히 맞아야 합니다. 한 번 반복할 때마다 두 요청에서 한 자리씩 늘어납니다.

## 요청 네 개를 받아도 되는지 결정하기

가중치와 기타 메모리를 모두 제외한 **순수 KV 예산 120 MiB**로 돌아갑니다. 요청마다 프롬프트 512자리 외에 최대 256자리를 추가로 저장하도록 예약합니다. 실제로 큰 배치를 할당할 필요 없이 방금 실측한 자리당 바이트 수로 계산할 수 있습니다.

In [ ]:
MiB = 1024**2
budget = 120 * MiB
requests, prompt, continuation = 4, 512, 256
for stage, length in [("prompt", prompt), ("reserved", prompt + continuation)]:
    per_request = length * bytes_per_position
    print(
        f"{stage}: {requests} requests use {requests * per_request / MiB:g} MiB; "
        f"{budget // per_request} requests fit"
    )

프롬프트 네 개는 90 MiB에 들어갑니다. 네 요청이 모두 예약한 길이까지 자라면 135 MiB가 필요하므로 끝까지 자랄 공간을 보장하려면 세 개까지 받을 수 있습니다. 처음부터 네 개를 모두 받으려면 동시에 길어졌을 때 일부 요청을 대기시키는 등의 정책이 필요합니다. 일찍 끝난 응답은 예약 공간을 전부 쓰지 않을 수도 있습니다.

이 계산은 CPU에서 전체 문맥을 유지하는 밀집 동적 캐시 한 벌의 키와 값만 셉니다. 프로세스 전체 메모리에는 가중치, 활성값, 텐서를 이어 붙일 때 쓰는 임시 버퍼, 할당기 부가 비용도 들어갑니다. 정적 캐시는 빈 위치를 미리 예약할 수 있고, 슬라이딩 윈도는 일부 층이 보관하는 과거 길이를 제한합니다. 페이지 단위 캐시는 블록에 맞춰 올림하거나 공통 접두사를 공유할 수 있습니다. 실제 서빙 용량을 정할 때는 해당 런타임의 텐서나 블록 할당기를 확인하세요.

### 참고 자료

- [실습에서 고정한 SmolLM2 모델 설정](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/93efa2f097d58c2a74874c7e644dbc9b0cee75a2/config.json)
- [Transformers의 캐시 설명](https://huggingface.co/docs/transformers/en/cache_explanation) · [실습 버전의 캐시 구현](https://github.com/huggingface/transformers/blob/v5.24.1/src/transformers/cache_utils.py)

[영어 원본](https://github.com/sungeuns/cookbook/blob/4640bbf5e5275380cf6a665e049378db30ae6d29/notebooks/en/measure_kv_cache_payload.ipynb)을 한국어로 옮긴 실습입니다. 이 노트북에는 Apache License 2.0이 적용되며 전문은 노트북 메타데이터에 들어 있습니다.